In [1]:
import sys, platform, torch
from importlib.metadata import version, PackageNotFoundError

print("Python", sys.version.split()[0], "|", platform.platform())
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| VRAM GB:", round(p.total_memory / 1e9, 1),
          "| compute capability:", f"{p.major}.{p.minor}")
    print("CUDA (torch build):", torch.version.cuda)
for pkg in ["transformers", "peft", "trl", "bitsandbytes", "datasets", "accelerate"]:
    try:
        print(pkg, version(pkg))
    except PackageNotFoundError:
        print(pkg, "NOT INSTALLED")

Python 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch 2.11.0+cu130 | CUDA available: True
GPU: Tesla T4 | VRAM GB: 15.6 | compute capability: 7.5
CUDA (torch build): 13.0
transformers 5.17.0
peft 0.21.0
trl NOT INSTALLED
bitsandbytes NOT INSTALLED
datasets 4.8.5
accelerate 1.15.0


In [2]:
!pip install -q bitsandbytes trl
!free -h
!df -h /content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 61.4 MB/s eta 0:00:00
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.5Gi       6.5Gi       2.3Mi       5.0Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   71G  38% /


In [3]:
import torch
from importlib.metadata import version
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

for pkg in ["transformers", "peft", "trl", "bitsandbytes", "accelerate"]:
    print(pkg, version(pkg))

name = "Qwen/Qwen2.5-0.5B-Instruct"   # small, ungated model, used only to test the stack
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map={"": 0})
print("allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

ids = tok("Paracetamol is used to", return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=20, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))


transformers 5.17.0
peft 0.21.0
trl 1.14.1
bitsandbytes 0.50.2
accelerate 1.15.0


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

allocated GB: 0.46
Paracetamol is used to treat pain and fever. It is a synthetic analgesic, which means that it works by blocking


In [4]:
ids = tok("Paracetamol is used to", return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=20, do_sample=False)
print(ids["input_ids"].shape, out.shape)
print(repr(tok.decode(out[0], skip_special_tokens=True)))

torch.Size([1, 7]) torch.Size([1, 27])
'Paracetamol is used to treat pain and fever. It is a synthetic analgesic, which means that it works by blocking'


In [5]:
import torch

prompt = "Paracetamol is used to"
ids = tok(prompt, return_tensors="pt").to("cuda")
print(ids["input_ids"][0].tolist())
print([tok.decode([t]) for t in ids["input_ids"][0]])

with torch.no_grad():
    logits = model(**ids).logits
print(logits.shape, logits.dtype)

probs = torch.softmax(logits[0, -1].float(), dim=-1)
print("sum of probs:", probs.sum().item())
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(repr(tok.decode([i])), round(p.item(), 4))

print("generate's first new token:", repr(tok.decode(out[0, 7:8])))


[4272, 67223, 309, 337, 374, 1483, 311]
['Par', 'acet', 'am', 'ol', ' is', ' used', ' to']
torch.Size([1, 7, 151936]) torch.bfloat16
sum of probs: 1.000000238418579
' treat' 0.4647
' relieve' 0.3194
' reduce' 0.0231
' help' 0.0204
' alleviate' 0.014
generate's first new token: ' treat'


In [6]:
from collections import Counter
print("config dtype:", getattr(model.config, "dtype", None) or getattr(model.config, "torch_dtype", None))
print("model.dtype:", model.dtype)
print(Counter(str(p.dtype) for p in model.parameters()))

config dtype: torch.bfloat16
model.dtype: torch.bfloat16
Counter({'torch.uint8': 168, 'torch.float16': 72, 'torch.bfloat16': 50})


In [7]:
import gc
from collections import Counter

del model
gc.collect(); torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    name, quantization_config=bnb, dtype=torch.float16, device_map={"": 0}
)
print(Counter(str(p.dtype) for p in model.parameters()))
print("allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

with torch.no_grad():
    logits = model(**ids).logits
print(logits.dtype, "| all finite:", torch.isfinite(logits).all().item())
probs = torch.softmax(logits[0, -1].float(), dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(repr(tok.decode([i])), round(p.item(), 4))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Counter({'torch.uint8': 168, 'torch.float16': 122})
allocated GB: 0.47
torch.float16 | all finite: True
' treat' 0.4597
' relieve' 0.3209
' reduce' 0.0255
' help' 0.0229
' alleviate' 0.0141


In [8]:
text = "Paracetamol is used to treat pain and fever."
enc = tok(text, return_tensors="pt").to("cuda")
ids_ = enc["input_ids"]

with torch.no_grad():
    res = model(**enc, labels=ids_)
print("HF loss:", res.loss.item())

logp = torch.log_softmax(res.logits[0, :-1].float(), dim=-1)
targets = ids_[0, 1:]
nll = -logp.gather(1, targets.unsqueeze(1)).squeeze(1)
for t, v in zip(targets, nll):
    print(repr(tok.decode([t])), round(v.item(), 3))
print("manual mean NLL:", nll.mean().item())

HF loss: 2.7911500930786133
'acet' 19.949
'am' 0.034
'ol' 0.033
' is' 0.866
' used' 3.014
' to' 0.711
' treat' 0.779
' pain' 2.189
' and' 1.047
' fever' 0.994
'.' 1.087
manual mean NLL: 2.7911503314971924


In [9]:
ctx = tok("Par", return_tensors="pt").to("cuda")
print(ctx["input_ids"])      # expect [[4272]], from your earlier output

with torch.no_grad():
    lg = model(**ctx).logits[0, -1].float()
pr = torch.softmax(lg, dim=-1)

top = torch.topk(pr, 8)
for p, i in zip(top.values, top.indices):
    print(repr(tok.decode([i])), round(p.item(), 4))

print("p('acet' | 'Par'):", pr[67223].item(),
      "| rank:", (pr > pr[67223]).sum().item() + 1)


tensor([[4272]], device='cuda:0')
'арат' 0.1711
'ame' 0.0176
'�' 0.0172
'ulan' 0.016
'奴隶' 0.0132
'erna' 0.0104
'arden' 0.0089
'олод' 0.0087
p('acet' | 'Par'): 2.1774155811726814e-09 | rank: 138003


In [10]:
words = ["paracetamol", "acetaminophen", "ibuprofen", "metformin",
         "hypertension", "fever", "myocardial infarction"]
for w in words:
    for form in (w, " " + w, " " + w.capitalize()):
        ids_ = tok(form, add_special_tokens=False)["input_ids"]
        pieces = [tok.decode([t]) for t in ids_]
        print(f"{form!r:32} {len(pieces)} tokens: {pieces}")

'paracetamol'                    4 tokens: ['par', 'acet', 'am', 'ol']
' paracetamol'                   4 tokens: [' par', 'acet', 'am', 'ol']
' Paracetamol'                   4 tokens: [' Par', 'acet', 'am', 'ol']
'acetaminophen'                  4 tokens: ['acet', 'amin', 'oph', 'en']
' acetaminophen'                 4 tokens: [' acet', 'amin', 'oph', 'en']
' Acetaminophen'                 5 tokens: [' Ac', 'et', 'amin', 'oph', 'en']
'ibuprofen'                      3 tokens: ['ib', 'upro', 'fen']
' ibuprofen'                     3 tokens: [' ib', 'upro', 'fen']
' Ibuprofen'                     3 tokens: [' Ib', 'upro', 'fen']
'metformin'                      3 tokens: ['met', 'form', 'in']
' metformin'                     3 tokens: [' met', 'form', 'in']
' Metformin'                     3 tokens: [' Met', 'form', 'in']
'hypertension'                   3 tokens: ['hyp', 'ert', 'ension']
' hypertension'                  1 tokens: [' hypertension']
' Hypertension'                  3 to

In [11]:
emb = model.get_input_embeddings()
print(emb.weight.shape, emb.weight.dtype)
print("hidden_size:", model.config.hidden_size,
      "| tied:", model.config.tie_word_embeddings)
out_emb = model.get_output_embeddings()
print("output head shares memory with embedding:",
      out_emb.weight.data_ptr() == emb.weight.data_ptr())

def vec(word):
    ids_ = tok(word, add_special_tokens=False)["input_ids"]
    assert len(ids_) == 1, f"{word!r} is {len(ids_)} tokens"
    return emb.weight[ids_[0]].float()

cos = torch.nn.functional.cosine_similarity
for a, b in [(" fever", " pain"), (" fever", " hypertension"),
             (" fever", " the"), (" pain", " the")]:
    print(a, "|", b, round(cos(vec(a), vec(b), dim=0).item(), 4))

torch.Size([151936, 896]) torch.float16
hidden_size: 896 | tied: True
output head shares memory with embedding: True
 fever |  pain 0.1329
 fever |  hypertension 0.2294
 fever |  the -0.1198
 pain |  the -0.0282


In [12]:
import torch, math
torch.manual_seed(0)
T, d = 4, 8
x = torch.randn(T, d)
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
Q, K, V = x @ Wq, x @ Wk, x @ Wv

scores = Q @ K.T / math.sqrt(d)
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
A = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)
out = A @ V

torch.set_printoptions(precision=3, sci_mode=False)
print(A)
print("row sums:", A.sum(dim=-1))
ref = torch.nn.functional.scaled_dot_product_attention(
    Q[None, None], K[None, None], V[None, None], is_causal=True)[0, 0]
print("max abs diff vs PyTorch:", (out - ref).abs().max().item())

tensor([[1.000, 0.000, 0.000, 0.000],
        [0.040, 0.960, 0.000, 0.000],
        [0.028, 0.972, 0.000, 0.000],
        [0.000, 0.000, 1.000, 0.000]])
row sums: tensor([1., 1., 1., 1.])
max abs diff vs PyTorch: 2.384185791015625e-07


In [13]:
c = model.config
keys = ["num_hidden_layers", "hidden_size", "num_attention_heads",
        "num_key_value_heads", "intermediate_size", "vocab_size"]
print({k: getattr(c, k, None) for k in keys})
print(model.model.layers[0])

{'num_hidden_layers': 24, 'hidden_size': 896, 'num_attention_heads': 14, 'num_key_value_heads': 2, 'intermediate_size': 4864, 'vocab_size': 151936}
Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
    (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
    (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
    (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [14]:
with torch.no_grad():
    z = model(**ids).logits[0, -1].float()

def entropy(p):
    return -(p * torch.log(p.clamp_min(1e-12))).sum().item()

for T in [0.2, 0.7, 1.0, 1.5]:
    p = torch.softmax(z / T, dim=-1)
    top = torch.topk(p, 3)
    shown = [(tok.decode([i]), round(v.item(), 4)) for v, i in zip(top.values, top.indices)]
    sp, _ = torch.sort(p, descending=True)
    n90 = (sp.cumsum(0) < 0.9).sum().item() + 1
    print(f"T={T}: entropy={entropy(p):.3f} nats | tokens for 90%: {n90} | {shown}")

T=0.2: entropy=0.409 nats | tokens for 90%: 2 | [(' treat', 0.8578), (' relieve', 0.1422), (' reduce', 0.0)]
T=0.7: entropy=0.966 nats | tokens for 90%: 2 | [(' treat', 0.5948), (' relieve', 0.3559), (' reduce', 0.0096)]
T=1.0: entropy=1.904 nats | tokens for 90%: 10 | [(' treat', 0.4597), (' relieve', 0.3209), (' reduce', 0.0255)]
T=1.5: entropy=4.930 nats | tokens for 90%: 791 | [(' treat', 0.1994), (' relieve', 0.1569), (' reduce', 0.029)]
